In [1]:
import os
import json
import logging
import pandas as pd
import numpy as np
from pprint import pprint
import nltk
from itertools import combinations
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import euclidean_distances


c:\Users\mkha1\Desktop\GQP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# NLTK

In [2]:

nltk.download('punkt')

section_types = ['TITLE', 'ABSTRACT', 'INTRO', 'FIG', 'METHODS', 'RESULTS']
directory = os.getcwd()
data_rows = []

# Compile regex patterns
fig_patterns = [
    re.compile(r'\bFig\.(?=\d)'), #full stop after "Fig" followed by a digit
    re.compile(r'\bet al\.'), #"et al."
    re.compile(r'\bFig\.\s([A-Z])'), #"Fig." followed by a capital letter
    re.compile(r'\bFig\.\s(\d)\((\w)\)'), #"Fig." followed by a digit and a letter in parentheses
    re.compile(r'\bFig\.\s(\d)') #"Fig." followed by a space and a digit
]

def process_passage(passage, section_type, base_filename):
    text = passage.get('text', '')
    # Replace the full stop after "Fig" followed by a digit with a space (e.g., "Fig. 4" -> "Fig 4")
    text = fig_patterns[0].sub(r'Fig \g<0>', text)  # Use "g<0>" to capture the matched string
    # Replace "et al." with "et al" (e.g., "Hsua et al. showed" -> "Hsua et al showed")
    text = fig_patterns[1].sub(r'et al ', text)
    # Replace "Fig." followed by a capital letter with "Fig" and the letter (e.g., "Fig. B" -> "Fig B")
    text = fig_patterns[2].sub(r'Fig \1', text)
    # Replace "Fig." followed by a digit and a letter in parentheses with "Fig" and the digit and letter (e.g., "Fig. 4(c)" -> "Fig 4(c)")
    text = fig_patterns[3].sub(r'Fig \1(\2)', text)
    # Replace "Fig." followed by a space and a digit (e.g., "Fig. 0" -> "Fig 0")
    text = fig_patterns[4].sub(r'Fig \1', text)


    # Split into sentences
    sentences = nltk.sent_tokenize(text)

    # Append each sentence as a new row in data_rows
    for sentence in sentences:
        data_rows.append({
            "filename": base_filename,
            "section_type": section_type,
            "sentence": sentence,
            "type": passage.get('infons', {}).get('type'),
        })

# Iterate over all JSON files in the directory
for filename in os.listdir(directory):
    if filename.endswith(".json"):
        logging.info(f"Processing file {filename}")
        try:
            with open(os.path.join(directory, filename), 'r') as file:
                data = json.load(file)
                base_filename = os.path.splitext(filename)[0]
                
                # Extract passages for each section type
                passages = data[0].get('documents', [{}])[0].get('passages', [])
                
                for section_type in section_types:
                    for passage in passages:
                        if (passage.get('infons', {}).get('section_type') == section_type and 
                                passage.get('infons', {}).get('type') != "title_1"):
                            process_passage(passage, section_type, base_filename)

        except json.JSONDecodeError as e:
            logging.error(f"Error parsing JSON file {filename}: {e}")
        except KeyError as e:
            logging.error(f"Error extracting text from JSON file {filename}: {e}")
        except Exception as e:
            logging.error(f"An unexpected error occurred while processing {filename}: {e}")

# Create a DataFrame from extracted rows if data_rows is not empty
if data_rows:
    df = pd.DataFrame(data_rows)
    logging.info(f"Extracted {len(df)} passages")
else:
    logging.info("No passages were extracted.")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mkha1\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


# Generate embeddings

In [ ]:
model = SentenceTransformer('microsoft/biogpt')

# Generate embeddings for the 'text' column
df['embeddings'] = df['sentence'].apply(lambda x: model.encode(x, show_progress_bar=False))

logging.info("Generated embeddings for all sentences.")

In [ ]:
embeddings_matrix = np.vstack(df['embeddings'].values) # Stack embeddings vertically to create a matrix suitable for further computations

# compute cosine similarity between different sections

### using cosine_similarity(embeddings_matrix) without itertools.combinations iteration was faster.

In [ ]:
# Initialize an empty similarity matrix filled with NaN instead of 0.0(not necessary)
n = len(embeddings_matrix)
similarity_matrix = np.full((n, n), np.nan)

# Use combinations to iterate through unique pairs of indices
for i, j in combinations(range(n), 2):
    # Calculate cosine similarity for the pair
    similarity = cosine_similarity(
        embeddings_matrix[i].reshape(1, -1),
        embeddings_matrix[j].reshape(1, -1)
    )[0, 0]
    
    # Populate the similarity matrix
    similarity_matrix[i, j] = similarity
    similarity_matrix[j, i] = similarity  # Assigning

# Use itertools.combinations to get section comparisons 

In [ ]:
# Add index so metadata match similarity matrix rows/columns
df['index'] = df.index

# Map section types to indexes
section_map = df.groupby('section_type')['index'].apply(list)

# Initialize  dictionary
section_comparisons = []

# Use itertools.combinations to iterate through section pairs
for (section_1, indices_1), (section_2, indices_2) in combinations(section_map.items(), 2):
    # Get the sub-matrix for the two sections
    sub_matrix = similarity_matrix[np.ix_(indices_1, indices_2)]
    
    # Calculate statistics for the section pair(not sure how if its done)
    mean_similarity = np.mean(sub_matrix)
    max_similarity = np.max(sub_matrix)
    
    # Append results
    section_comparisons.append({
        'Section 1': section_1,
        'Section 2': section_2,
        'Mean Similarity': mean_similarity,
        'Max Similarity': max_similarity
    })

# Get a DataFrame
comparison_df = pd.DataFrame(section_comparisons)

# Same for euclidean 

In [ ]:
# Initialize an empty distance matrix filled with NaN instead of 0.0(not necessary)
n = len(embeddings_matrix)
distance_matrix = np.full((n, n), np.nan)

# Use combinations to iterate through unique pairs of indices
for i, j in combinations(range(n), 2):
    # Calculate cosine similarity for the pair
    distance = euclidean_distances(
        embeddings_matrix[i].reshape(1, -1),
        embeddings_matrix[j].reshape(1, -1)
    )[0, 0]
    
    # Populate the distance matrix
    distance_matrix[i, j] = distance
    distance_matrix[j, i] = distance  # Assigning
# Add an index to the metadata to match the distance matrix rows/columns

df['index'] = df.index

# Map section types to indexes
section_map = df.groupby('section_type')['index'].apply(list)

# Initialize a dictionary to store results
section_distance = []

# Use itertools.combinations to iterate through unique section pairs
for (section_1, indices_1), (section_2, indices_2) in combinations(section_map.items(), 2):
    # Get the sub-matrix for the two sections
    sub_matrix = distance_matrix[np.ix_(indices_1, indices_2)]
    
    # Calculate statistics for this section pair
    mean_distance = np.mean(sub_matrix)
    max_distance = np.max(sub_matrix)
    
    # Append the results as a dictionary
    section_distance.append({
        'Section 1': section_1,
        'Section 2': section_2,
        'Mean distance': mean_distance,
        'Max distance': max_distance
    })

# Convert the results into a DataFrame
distance_df = pd.DataFrame(section_distance)